[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/dlt-certified/blob/main/notebooks/day-04-incremental.ipynb)

---
# Day 4 · Incremental Loading and State Management
**certified-journeys / dlt-certified** &nbsp;|&nbsp; Efficiency at Scale

> **Goal for today:** Configure incremental loading with `dlt.sources.incremental`, understand cursor fields and `last_value`, inspect how dlt persists pipeline state in the destination, and build a source that only fetches new records on each run.

---
## Why incremental loading matters

Full refresh (reloading every record on every run) works fine for small datasets, but it breaks down fast:

| Approach | 1 M rows | 100 M rows | 1 B rows |
|---|---|---|---|
| Full refresh | ~2 min | ~3 hours | days |
| Incremental | ~2 sec | ~2 sec | ~2 sec |

Incremental loading means: **only fetch records newer than the last run**.

### The cursor pattern

Every record has a field that increases monotonically as records are created or updated: `updated_at`, `created_at`, `id`, `sequence_number`. dlt calls this the **cursor field**.

```
Run 1: fetch all records where updated_at > 1970-01-01
       → loads 1000 records
       → stores last_value = 2024-01-15T09:00:00

Run 2: fetch all records where updated_at > 2024-01-15T09:00:00
       → loads 12 new records
       → stores last_value = 2024-01-15T10:35:00
```

dlt automates the storage and retrieval of `last_value` — you just declare the cursor.

---
## How dlt state works

dlt stores pipeline state (including the cursor `last_value`) **inside the destination itself** — in a special `_dlt_pipeline_state` table. This means:

- No external Redis, S3, or database needed for state.
- State is automatically consistent with the data (both updated in one transaction).
- State survives process restarts, container restarts, and re-deployments.
- You can inspect state by querying the destination directly.

```
DuckDB file
├── dataset_name
│   ├── orders          ← your data
│   ├── _dlt_loads      ← load history
│   ├── _dlt_pipeline_state  ← cursor last_value + any custom state
│   └── _dlt_version    ← schema version history
```

---
## Step 1 · Install dlt

In [ ]:
%pip install -q "dlt[duckdb]"

---
## Step 2 · Build a simulated paginated API with timestamps

We'll simulate an API that returns records ordered by `updated_at`. In a real pipeline this would be an HTTP call. The simulation lets us verify behaviour without network calls.

In [ ]:
import dlt
from datetime import datetime, timezone

# --- Simulated API data -------------------------------------------------
# Represents records already in the API on day 1
INITIAL_RECORDS = [
    {"id": 1, "name": "Record A", "updated_at": "2024-01-10T08:00:00Z"},
    {"id": 2, "name": "Record B", "updated_at": "2024-01-11T09:30:00Z"},
    {"id": 3, "name": "Record C", "updated_at": "2024-01-12T14:15:00Z"},
    {"id": 4, "name": "Record D", "updated_at": "2024-01-13T07:45:00Z"},
    {"id": 5, "name": "Record E", "updated_at": "2024-01-14T16:00:00Z"},
]

# Records that "arrive" after the first run (higher updated_at)
NEW_RECORDS = [
    {"id": 6, "name": "Record F", "updated_at": "2024-01-15T10:00:00Z"},
    {"id": 7, "name": "Record G", "updated_at": "2024-01-16T11:30:00Z"},
    {"id": 3, "name": "Record C (updated)", "updated_at": "2024-01-16T12:00:00Z"},  # id=3 re-appears with a later timestamp
]

# This global simulates which records the API would return
# (switched between runs below)
ALL_API_RECORDS = INITIAL_RECORDS


def fake_api_call(since: str):
    """
    Simulates an API that accepts a 'since' timestamp and returns
    only records with updated_at > since, sorted ascending.
    """
    return sorted(
        [r for r in ALL_API_RECORDS if r["updated_at"] > since],
        key=lambda r: r["updated_at"]
    )


print("Simulated API ready.")
print(f"Initial records: {len(INITIAL_RECORDS)}")
print(f"New records (run 2): {len(NEW_RECORDS)}")

---
## Step 3 · Build an incremental resource with `dlt.sources.incremental`

The `dlt.sources.incremental` object is injected into your resource function as a parameter. It carries:
- `last_value` — the highest cursor value seen in previous runs (or `initial_value` on first run)
- Automatic filtering — dlt can filter records for you so you don't have to

On first run: `last_value = initial_value` (you set this — often `"1970-01-01T00:00:00Z"` or `0`).

In [ ]:
@dlt.resource(
    name="records",
    primary_key="id",
    write_disposition="merge",  # merge so updated records (id=3) overwrite old ones
)
def incremental_records(
    # dlt injects this parameter automatically when it sees dlt.sources.incremental
    updated_at=dlt.sources.incremental(
        cursor_path="updated_at",           # which field in each record is the cursor
        initial_value="1970-01-01T00:00:00Z", # start from the beginning on first run
    )
):
    """
    Yields records with updated_at > last_value.
    On first run, last_value = initial_value (i.e., all records).
    On subsequent runs, last_value = highest updated_at seen previously.
    """
    print(f"  → Fetching records with updated_at > '{updated_at.last_value}'")

    # Call the simulated API with the cursor value
    records = fake_api_call(since=updated_at.last_value)

    print(f"  → API returned {len(records)} records")
    yield from records


print("Resource defined.")

---
## Step 4 · Run 1 — load all historical records

In [ ]:
# Ensure the simulated API only has the initial records
ALL_API_RECORDS = INITIAL_RECORDS

pipeline = dlt.pipeline(
    pipeline_name="incremental_demo",
    destination="duckdb",
    dataset_name="inc",
)

print("=== Run 1 ===")
info1 = pipeline.run(incremental_records())

with pipeline.sql_client() as client:
    with client.execute_query(
        "SELECT id, name, updated_at FROM inc.records ORDER BY updated_at"
    ) as cur:
        df = cur.df()
        print(f"\nLoaded {len(df)} rows:")
        print(df.to_string(index=False))

**What just happened?**
- `last_value` was `"1970-01-01T00:00:00Z"` (initial value), so the API returned all 5 records.
- dlt stored the highest `updated_at` seen (`2024-01-14T16:00:00Z`) as the new `last_value` in state.
- On the next run, the cursor will start from there.

---
## Step 5 · Inspect pipeline state after run 1

In [ ]:
# Method 1: Access state via the Python API
state = pipeline.state
print("Pipeline state (Python dict):")

# The incremental cursor is stored under sources -> resource name -> incremental
import json
print(json.dumps(state, indent=2, default=str))

In [ ]:
# Method 2: Query the _dlt_pipeline_state table directly in DuckDB
# This shows what's persisted in the destination (survives process restarts)
with pipeline.sql_client() as client:
    with client.execute_query(
        "SELECT pipeline_name, created_at FROM inc._dlt_pipeline_state ORDER BY created_at DESC LIMIT 3"
    ) as cur:
        print("State rows in DuckDB:")
        print(cur.df().to_string(index=False))

---
## Step 6 · Run 2 — only new records load

In [ ]:
# Simulate new records arriving at the API
ALL_API_RECORDS = INITIAL_RECORDS + NEW_RECORDS

print("=== Run 2 ===")
print(f"(Simulated API now has {len(ALL_API_RECORDS)} records total)")
info2 = pipeline.run(incremental_records())

with pipeline.sql_client() as client:
    with client.execute_query(
        "SELECT id, name, updated_at FROM inc.records ORDER BY updated_at"
    ) as cur:
        df2 = cur.df()
        print(f"\nFinal table: {len(df2)} rows")
        print(df2.to_string(index=False))

    print("\n── Row count summary ──")
    with client.execute_query("SELECT COUNT(*) AS n FROM inc.records") as cur:
        print(f"Total rows: {cur.df()['n'][0]}")

**What just happened?**
- Run 2 started with `last_value = "2024-01-14T16:00:00Z"` (persisted from run 1).
- The API returned only the 3 records with `updated_at > "2024-01-14T16:00:00Z"` — records F, G, and the updated Record C.
- `write_disposition="merge"` with `primary_key="id"` ensured Record C (id=3) was **updated in place**, not duplicated.
- Final row count: 7 (5 original + 2 new; Record C replaced its old version).
- The whole second run only processed 3 records — not 8.

---
## Step 7 · Verify idempotency — run 2 again with the same data

In [ ]:
print("=== Run 3 (same data as run 2) ===")
info3 = pipeline.run(incremental_records())

with pipeline.sql_client() as client:
    with client.execute_query("SELECT COUNT(*) AS n FROM inc.records") as cur:
        count3 = cur.df()['n'][0]
        print(f"Row count after run 3: {count3}  ← should still be 7")

print("\nNo new records = cursor doesn't advance further.")
print(f"State cursor: {pipeline.state.get('sources', {}).get('incremental_records', {})}")

---
## Step 8 · Incremental by integer ID

Some APIs don't have timestamps — they use auto-incrementing integer IDs. dlt handles this identically.

In [ ]:
# Simulate an API that returns records with integer IDs
ID_BATCH_1 = [{"id": i, "val": f"item_{i}"} for i in range(1, 6)]   # IDs 1-5
ID_BATCH_2 = [{"id": i, "val": f"item_{i}"} for i in range(6, 10)]  # IDs 6-9

ID_API_DATA = ID_BATCH_1  # start with batch 1


def fetch_by_id(since_id: int):
    return [r for r in ID_API_DATA if r["id"] > since_id]


@dlt.resource(
    name="items_by_id",
    primary_key="id",
    write_disposition="append",  # append is fine here — IDs never repeat
)
def items_incremental(
    item_id=dlt.sources.incremental(
        cursor_path="id",
        initial_value=0,  # start from ID 0 on first run
    )
):
    print(f"  → Fetching items with id > {item_id.last_value}")
    yield from fetch_by_id(since_id=item_id.last_value)


p_id = dlt.pipeline(
    pipeline_name="id_incremental",
    destination="duckdb",
    dataset_name="id_demo",
)

# Run 1
ID_API_DATA = ID_BATCH_1
p_id.run(items_incremental())

with p_id.sql_client() as client:
    with client.execute_query("SELECT COUNT(*) AS n FROM id_demo.items_by_id") as cur:
        print(f"After run 1: {cur.df()['n'][0]} rows")

# Run 2 — new items arrive
ID_API_DATA = ID_BATCH_1 + ID_BATCH_2
p_id.run(items_incremental())

with p_id.sql_client() as client:
    with client.execute_query("SELECT COUNT(*) AS n FROM id_demo.items_by_id") as cur:
        print(f"After run 2: {cur.df()['n'][0]} rows  ← only 4 new items loaded")
    with client.execute_query("SELECT id, val FROM id_demo.items_by_id ORDER BY id") as cur:
        print(cur.df().to_string(index=False))

---
## Step 9 · Why incremental beats full refresh — a concrete comparison

In [ ]:
import time

# Simulate a larger dataset: 10,000 records
BIG_DATASET = [
    {"id": i, "val": f"row_{i}", "updated_at": f"2024-01-{(i % 28)+1:02d}T00:00:00Z"}
    for i in range(1, 10_001)
]
# 50 new records that arrived after the last run
NEW_50 = [
    {"id": i, "val": f"row_{i}", "updated_at": f"2024-02-{(i % 28)+1:02d}T00:00:00Z"}
    for i in range(10_001, 10_051)
]

# Full refresh resource
@dlt.resource(name="big_table", primary_key="id", write_disposition="replace")
def full_refresh():
    yield BIG_DATASET + NEW_50  # always loads everything

# Incremental resource
@dlt.resource(name="big_table_inc", primary_key="id", write_disposition="append")
def incremental(updated_at=dlt.sources.incremental("updated_at", initial_value="2024-01-28T00:00:00Z")):
    # Pretend the cursor is already at end of BIG_DATASET — second run scenario
    yield from [r for r in (BIG_DATASET + NEW_50) if r["updated_at"] > updated_at.last_value]

p_compare = dlt.pipeline(pipeline_name="compare_strategies", destination="duckdb", dataset_name="cmp")

# Time the full refresh
t0 = time.time()
p_compare.run(full_refresh())
t_full = time.time() - t0
rows_full = 10_050  # always processes everything

# Time the incremental (simulates run 2 — only new records)
t0 = time.time()
p_compare.run(incremental())
t_inc = time.time() - t0

print("=== Strategy Comparison ===")
print(f"Full refresh : processed 10,050 records  in {t_full:.2f}s")
print(f"Incremental  : processed      50 records  in {t_inc:.3f}s")
print(f"Speed-up     : ~{t_full/max(t_inc, 0.001):.0f}x faster")

---
## Challenge — do this yourself

Build an incremental source that simulates a ticket system:

1. Each ticket has `ticket_id` (int), `title` (str), `status` (str), and `updated_at` (ISO timestamp string).
2. Use `dlt.sources.incremental("updated_at", initial_value="2024-01-01T00:00:00Z")`.
3. Use `write_disposition="merge"` with `primary_key="ticket_id"`.
4. Run 1: load 4 tickets (all from January 2024).
5. Run 2: load a batch where one ticket's status changed (new `updated_at`) and one new ticket was added.
6. After run 2, print: total rows, and which tickets have `status="closed"`.

**Expected behaviour:** Run 2 should only fetch and process the 2 changed/new records, not all 4.

In [ ]:
# Your solution here


<details>
<summary>Show solution</summary>

```python
import dlt

TICKETS_V1 = [
    {"ticket_id": 1, "title": "Login bug",     "status": "open",   "updated_at": "2024-01-05T10:00:00Z"},
    {"ticket_id": 2, "title": "UI glitch",     "status": "open",   "updated_at": "2024-01-08T14:00:00Z"},
    {"ticket_id": 3, "title": "Slow API",      "status": "open",   "updated_at": "2024-01-10T09:00:00Z"},
    {"ticket_id": 4, "title": "Wrong total",   "status": "closed", "updated_at": "2024-01-12T16:00:00Z"},
]

TICKETS_V2 = [
    {"ticket_id": 2, "title": "UI glitch",     "status": "closed", "updated_at": "2024-01-20T10:00:00Z"},  # closed
    {"ticket_id": 5, "title": "Missing data",  "status": "open",   "updated_at": "2024-01-21T08:00:00Z"},  # new
]

API_DATA = TICKETS_V1

def api_tickets(since):
    return [t for t in API_DATA if t["updated_at"] > since]

@dlt.resource(name="tickets", primary_key="ticket_id", write_disposition="merge")
def tickets(updated_at=dlt.sources.incremental("updated_at", initial_value="2024-01-01T00:00:00Z")):
    batch = api_tickets(updated_at.last_value)
    print(f"  → API returned {len(batch)} records")
    yield from batch

p = dlt.pipeline(pipeline_name="tickets_demo", destination="duckdb", dataset_name="tk")

API_DATA = TICKETS_V1
p.run(tickets())

API_DATA = TICKETS_V1 + TICKETS_V2
p.run(tickets())

with p.sql_client() as client:
    with client.execute_query("SELECT COUNT(*) AS n FROM tk.tickets") as cur:
        print(f"Total rows: {cur.df()['n'][0]}")  # 5
    with client.execute_query("SELECT ticket_id, title, status FROM tk.tickets WHERE status='closed'") as cur:
        print("Closed tickets:")
        print(cur.df().to_string(index=False))
```
</details>

---
## Day 4 key concepts recap

| Concept | What to remember |
|---|---|
| `dlt.sources.incremental` | Decorator argument that declares a cursor field — dlt manages `last_value` automatically |
| `cursor_path` | The field name in each record that acts as the watermark |
| `initial_value` | Starting cursor for the very first run — typically epoch or `0` |
| `last_value` | The highest cursor value seen on the last run — stored in destination |
| State persistence | Stored in `_dlt_pipeline_state` table in the destination — no external DB needed |
| `merge` + incremental | The canonical combo: only new/updated records fetched AND upserted |
| Idempotency | Running the same incremental run twice produces the same result |
| Why incremental wins | Processes only changed records — seconds instead of hours at scale |

> **Tip:** Always prefer incremental loading over full refresh. dlt state is stored in the destination — no external DB needed. The `merge` + `incremental` combination is the gold standard for production pipelines.

---
## What's next → Day 5

**Day 5** → Schema management and data contracts: column hints, schema evolution, `freeze` contracts, and how to lock down a production pipeline against unexpected data changes.

Mark Day 4 complete in your [tracker](../index.html).